In [1]:
!git clone https://github.com/txst-cs-smartfall/SmartFallMM-Dataset

Cloning into 'SmartFallMM-Dataset'...
remote: Enumerating objects: 26033, done.
remote: Counting objects: 100% (2960/2960), done.
remote: Compressing objects: 100% (2854/2854), done.
remote: Total 26033 (delta 128), reused 2463 (delta 106), pack-reused 23073 (from 1)
Receiving objects: 100% (26033/26033), 533.28 MiB | 18.25 MiB/s, done.
Resolving deltas: 100% (972/972), done.
Updating files: 100% (22614/22614), done.


In [2]:
!python convert_smartfallmm.py --repo SmartFallMM-Dataset --out my_data

/content/convert_smartfallmm.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(raw, errors="coerce")
/content/convert_smartfallmm.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(raw, errors="coerce")
/content/convert_smartfallmm.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dts = pd.to_datetime(raw, errors="coerce")
/content/convert_smartfallmm.py:45: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please spe

In [3]:
%%writefile fall_pipeline_pico.py
import argparse
import glob
import os
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

TARGET_HZ = 100   # MPU6050 sampling rate on the Pico 2 W
SRC_HZ = 100      # Fallback rate if CSV has no 't' column
WIN_S = 3.0       # 3-second window (300 samples)
STEP_S = 1.0      # 1-second sliding step for ADL windows
FREEFALL_G = 0.6  # Free-fall threshold in g

FEATURE_NAMES = [
    "svm_max", "svm_min", "svm_mean", "svm_std", "freefall_s",
    "jerk_max", "tilt_change_deg", "post_std", "gyro_max", "gyro_mean"
]


def load_trial(path):
    """Read one CSV and linearly interpolate to TARGET_HZ (100 Hz)."""
    df = pd.read_csv(path)
    cols = [c for c in ["ax", "ay", "az", "gx", "gy", "gz"] if c in df.columns]
    if not {"ax", "ay", "az"}.issubset(cols):
        return None
    x = df[cols].values.astype(float)
    if len(x) < 10:
        return None

    if "t" in df.columns and len(df) > 2:
        t = df["t"].values.astype(float)
        t = t - t[0]
    else:
        t = np.arange(len(x)) / float(SRC_HZ)

    duration = t[-1]
    if duration < 1.0:
        return None

    t_new = np.arange(0, duration, 1.0 / TARGET_HZ)
    x_new = np.zeros((len(t_new), x.shape[1]))
    for i in range(x.shape[1]):
        x_new[:, i] = np.interp(t_new, t, x[:, i])

    if x_new.shape[1] == 3:
        x_new = np.hstack([x_new, np.zeros((len(x_new), 3))])
    return x_new


def windows_from_trial(x, is_fall):
    """Fall -> 3 shifted windows around impact peak. ADL -> 1s sliding windows."""
    w = int(WIN_S * TARGET_HZ)
    if len(x) < TARGET_HZ:
        return []
    if len(x) < w:
        x = np.pad(x, ((0, w - len(x)), (0, 0)), mode="edge")
    svm = np.linalg.norm(x[:, :3], axis=1)
    if is_fall:
        c = int(np.argmax(svm))
        wins = []
        for offset in [w // 4, w // 3, w // 2]:
            s = max(0, min(c - offset, len(x) - w))
            wins.append(x[s:s + w])
        return wins
    step = int(STEP_S * TARGET_HZ)
    return [x[i:i + w] for i in range(0, len(x) - w + 1, step)]


def features(win):
    """Extract 10 physics features from one 3-second window [300, 6]."""
    a, g = win[:, :3], win[:, 3:]
    svm = np.linalg.norm(a, axis=1)
    gmag = np.linalg.norm(g, axis=1)
    p = int(np.argmax(svm))
    dt = 1.0 / TARGET_HZ
    jerk = np.abs(np.diff(svm)) / dt

    pre_start = max(0, p - 50)
    pre_end = max(pre_start + 1, p - 10)
    pre = a[pre_start:pre_end].mean(axis=0)

    post = a[p + 30:].mean(axis=0) if p + 30 < len(a) else a[-1]
    cosang = np.dot(pre, post) / (np.linalg.norm(pre) * np.linalg.norm(post) + 1e-9)
    tilt_change = np.degrees(np.arccos(np.clip(cosang, -1.0, 1.0)))
    after = svm[min(len(svm) - 1, p + 15):]

    return [
        float(svm.max()),
        float(svm.min()),
        float(svm.mean()),
        float(svm.std()),
        float((svm < FREEFALL_G).sum() * dt),
        float(jerk.max()),
        float(tilt_change),
        float(after.std()) if len(after) > 1 else 0.0,
        float(gmag.max()),
        float(gmag.mean()),
    ]


def build_dataset(folder):
    X, y, groups = [], [], []
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not files:
        raise SystemExit(f"No .csv files found in '{folder}'.")
    for path in files:
        parts = os.path.basename(path).split("_")
        if len(parts) < 2:
            continue
        subject, label = parts[0], parts[1].lower()
        x = load_trial(path)
        if x is None:
            continue
        is_fall = label.startswith("fall")
        for w in windows_from_trial(x, is_fall):
            X.append(features(w))
            y.append(int(is_fall))
            groups.append(subject)
    return np.array(X), np.array(y), np.array(groups)


def make_models():
    return {
        "Random Forest (Pico 2 W model)": RandomForestClassifier(
            n_estimators=35, max_depth=8, min_samples_leaf=2,
            class_weight="balanced_subsample", random_state=0, n_jobs=-1),
        "SVM (comparison)": make_pipeline(
            StandardScaler(), SVC(kernel="rbf", class_weight="balanced")),
    }


def evaluate_and_report(X, y, groups, threshold=0.30, png="confusion_matrix.png"):
    gkf = GroupKFold(n_splits=min(5, len(set(groups))))
    results = {}
    for name in make_models():
        y_true, y_pred, y_prob = [], [], []
        for tr, te in gkf.split(X, y, groups):
            model = make_models()[name]
            model.fit(X[tr], y[tr])
            y_true += list(y[te])
            y_pred += list(model.predict(X[te]))
            if hasattr(model, "predict_proba"):
                y_prob += list(model.predict_proba(X[te])[:, 1])
        results[name] = (np.array(y_true), np.array(y_pred),
                         np.array(y_prob) if y_prob else None)

    rf_name = "Random Forest (Pico 2 W model)"
    yt_rf, _, prob = results[rf_name]
    n_adl = int((yt_rf == 0).sum())
    hours = max(0.01, n_adl * STEP_S / 3600.0)

    print(f"\n=== SUBJECT-WISE CROSS-VALIDATION RESULTS (Threshold = {threshold}) ===")
    print(f"{'Model':32s} {'Accuracy':>9s} {'Sensitivity':>12s} {'Specificity':>12s}")
    for name, (yt, yp, pr) in results.items():
        if pr is not None:
            yp = (pr >= threshold).astype(int)
        print(f"{name:32s} {accuracy_score(yt, yp):9.3f} "
              f"{recall_score(yt, yp, pos_label=1):12.3f} {recall_score(yt, yp, pos_label=0):12.3f}")

    print("\nThreshold Tuning Table (Pick a threshold where Sensitivity >= 0.90):")
    print(f"{'threshold':>9s} {'accuracy':>10s} {'sensitivity':>12s} {'specificity':>12s} {'false-alarms/hr':>17s}")
    for th in [0.50, 0.45, 0.40, 0.35, 0.30, 0.25, 0.20]:
        yp = (prob >= th).astype(int)
        fp = int(((yp == 1) & (yt_rf == 0)).sum())
        print(f"{th:9.2f} {accuracy_score(yt_rf, yp):10.3f} {recall_score(yt_rf, yp, pos_label=1):12.3f} "
              f"{recall_score(yt_rf, yp, pos_label=0):12.3f} {fp / hours:17.1f}")

    yp_final = (prob >= threshold).astype(int)
    cm = confusion_matrix(yt_rf, yp_final, labels=[0, 1])
    print(f"\nConfusion Matrix at threshold={threshold} (rows=True, cols=Predicted):")
    print(f"               pred not-fall   pred fall")
    print(f"true not-fall  {cm[0, 0]:13d}  {cm[0, 1]:10d}")
    print(f"true fall      {cm[1, 0]:13d}  {cm[1, 1]:10d}")

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4, 3.5))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks([0, 1], ["not fall", "fall"])
    ax.set_yticks([0, 1], ["not fall", "fall"])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")
    fig.tight_layout()
    fig.savefig(png, dpi=150)
    print(f"Saved portfolio image: {png}")


def export_to_micropython(rf_model, threshold, out_py="fall_model_pico.py"):
    """Compile scikit-learn RandomForest into pure MicroPython if/else trees."""
    lines = [
        "# Auto-generated Random Forest for Raspberry Pi Pico 2 W (MicroPython)",
        f"THRESHOLD = {threshold:.4f}",
        f"N_TREES = {len(rf_model.estimators_)}",
        ""
    ]
    for idx, est in enumerate(rf_model.estimators_):
        tree = est.tree_
        lines.append(f"def _tree_{idx}(f):")
        def recurse(node, depth):
            indent = "    " * depth
            if tree.feature[node] != -2:
                feat_idx = int(tree.feature[node])
                th_val = float(tree.threshold[node])
                lines.append(f"{indent}if f[{feat_idx}] <= {th_val:.6f}:")
                recurse(tree.children_left[node], depth + 1)
                lines.append(f"{indent}else:")
                recurse(tree.children_right[node], depth + 1)
            else:
                vals = tree.value[node][0]
                prob_fall = float(vals[1] / (vals[0] + vals[1] + 1e-9))
                lines.append(f"{indent}return {prob_fall:.4f}")
        recurse(0, 1)
        lines.append("")

    tree_calls = " + ".join([f"_tree_{i}(f)" for i in range(len(rf_model.estimators_))])
    lines.append("def predict_fall_prob(f):")
    lines.append(f"    return ({tree_calls}) / N_TREES")
    lines.append("")

    with open(out_py, "w") as fp:
        fp.write("\n".join(lines))
    print(f"\nSaved MicroPython model: '{out_py}' ({os.path.getsize(out_py) // 1024} KB)")
    print("Feature Importances (for your portfolio rubric):")
    for n, v in sorted(zip(FEATURE_NAMES, rf_model.feature_importances_), key=lambda t: -t[1]):
        print(f"  {n:18s} {v:.3f}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="Folder with converted CSV files")
    ap.add_argument("--threshold", type=float, default=0.35, help="Decision threshold")
    ap.add_argument("--out", default="fall_model_pico.py", help="MicroPython output file")
    args = ap.parse_args()

    X, y, groups = build_dataset(args.data)
    print(f"Dataset summary -> Windows: {len(y)} | Falls: {int(y.sum())} | ADLs: {int((y==0).sum())} | Subjects: {len(set(groups))}")
    evaluate_and_report(X, y, groups, args.threshold)

    rf = make_models()["Random Forest (Pico 2 W model)"]
    rf.fit(X, y)
    export_to_micropython(rf, args.threshold, args.out)

Writing fall_pipeline_pico.py


In [4]:
!python fall_pipeline_pico.py --data my_data --threshold 0.35

Dataset summary -> Windows: 72459 | Falls: 1893 | ADLs: 70566 | Subjects: 57

=== SUBJECT-WISE CROSS-VALIDATION RESULTS (Threshold = 0.35) ===
Model                             Accuracy  Sensitivity  Specificity
Random Forest (Pico 2 W model)       0.945        0.877        0.947
SVM (comparison)                     0.950        0.822        0.954

Threshold Tuning Table (Pick a threshold where Sensitivity >= 0.90):
threshold   accuracy  sensitivity  specificity   false-alarms/hr
     0.50      0.969        0.840        0.972             101.0
     0.45      0.963        0.852        0.966             122.4
     0.40      0.956        0.866        0.958             150.4
     0.35      0.945        0.877        0.947             192.6
     0.30      0.932        0.893        0.933             240.6
     0.25      0.917        0.910        0.918             296.7
     0.20      0.899        0.927        0.898             366.5

Confusion Matrix at threshold=0.35 (rows=True, cols=Predict

In [5]:
!python fall_pipeline_pico.py --data my_data --threshold 0.25

Dataset summary -> Windows: 72459 | Falls: 1893 | ADLs: 70566 | Subjects: 57

=== SUBJECT-WISE CROSS-VALIDATION RESULTS (Threshold = 0.25) ===
Model                             Accuracy  Sensitivity  Specificity
Random Forest (Pico 2 W model)       0.917        0.910        0.918
SVM (comparison)                     0.950        0.822        0.954

Threshold Tuning Table (Pick a threshold where Sensitivity >= 0.90):
threshold   accuracy  sensitivity  specificity   false-alarms/hr
     0.50      0.969        0.840        0.972             101.0
     0.45      0.963        0.852        0.966             122.4
     0.40      0.956        0.866        0.958             150.4
     0.35      0.945        0.877        0.947             192.6
     0.30      0.932        0.893        0.933             240.6
     0.25      0.917        0.910        0.918             296.7
     0.20      0.899        0.927        0.898             366.5

Confusion Matrix at threshold=0.25 (rows=True, cols=Predict